# 01 · Data profiling

Checks on the raw loads and the typed staging layer before any analysis:
provenance, row counts, the cohort flow, suppression/null rates and the
distributions that drive cohort rules (volume, payments).

Set `PARTD_TARGET` (and `DUCKDB_PATH` for DuckDB) before starting Jupyter.

In [ ]:
from _setup import CFG, WH, banner, q, table

banner()

## Provenance and row counts

In [ ]:
WH.read("reporting", "rpt_data_provenance")[
    ["source_name", "row_count", "source_url", "downloaded_at", "is_synthetic"]
]

In [ ]:
counts = WH.read("reporting", "rpt_source_row_counts")
counts.assign(millions=counts["row_count"] / 1e6).sort_values("row_count", ascending=False)

## Cohort flow

In [ ]:
WH.read("reporting", "rpt_cohort_flow")

## Null / suppression rates in the prescriber-level file

CMS blanks cells below 11. Case-mix controls with high null rates get a
missingness indicator in the peer-adjustment regression.

In [ ]:
cols = [
    "total_beneficiaries", "bene_avg_risk_score", "bene_avg_age", "bene_dual_count",
    "lis_claims", "opioid_claims", "opioid_long_acting_claims", "cms_brand_claims",
]  # fmt: skip
null_sql = ",\n".join(f"avg(case when {c} is null then 1.0 else 0 end) as {c}" for c in cols)
q(f"select {null_sql} from {table('staging', 'stg_part_d__prescriber')}").T.rename(
    columns={0: "null_rate"}
)

## Prescribing volume (drives the `min_total_fills` rule)

In [ ]:
fills = q(f"select drug_detail_fills from {table('marts', 'fct_prescribers')}")
ax = fills["drug_detail_fills"].clip(upper=5000).plot.hist(bins=100)
ax.axvline(100, color="black", linewidth=1)
ax.set_xlabel("30-day fills in the drug-level file (clipped at 5,000)")
ax.set_title("Prescriber volume; line = cohort threshold")

## Industry payments

In [ ]:
payments = q(
    f"""
    select payment_tier, count(*) as n, sum(total_payment_usd) as usd
    from {table('marts', 'fct_prescribers')}
    where in_cohort
    group by 1 order by 1
    """
)
payments.assign(share=payments["n"] / payments["n"].sum())

In [ ]:
q(
    f"""
    select payment_category_mix.*
    from (
        select 'food_beverage' as category, sum(food_beverage_usd) as usd from {table('marts', 'fct_prescribers')}
        union all select 'speaker_faculty', sum(speaker_faculty_usd) from {table('marts', 'fct_prescribers')}
        union all select 'consulting', sum(consulting_usd) from {table('marts', 'fct_prescribers')}
        union all select 'travel_lodging', sum(travel_lodging_usd) from {table('marts', 'fct_prescribers')}
    ) as payment_category_mix
    order by usd desc
    """
)

## Multi-source drugs and national brand premium

In [ ]:
drugs = WH.read("marts", "dim_drugs")
print(f"{len(drugs):,} generic names; {drugs['is_multisource'].sum():,} multi-source")
drugs[drugs["is_multisource"]].nlargest(15, "brand_fills")[
    ["generic_name", "example_brand_name", "brand_fills", "generic_fills",
     "brand_cost_per_fill", "generic_cost_per_fill", "national_brand_share"]
]  # fmt: skip

## Later exclusions in the cohort

In [ ]:
WH.read("reporting", "rpt_exclusion_outcomes").sort_values(["exclusion_year", "n_prescribers"])

In [ ]:
print(f"Data year {CFG.data_year}; outcome window starts {CFG.data_year + 1}-01-01")